# 00 — Colab Setup / ตั้งค่า Colab

**EN.** First-cell bootstrap that mounts Google Drive, clones the HeatShield repo using `GH_PAT` from Colab Secrets, symlinks `data/` and `app/models/forecast_v3/` to Drive, and installs `requirements-train.txt`. Idempotent — safe to re-run.

**TH.** เซลล์ตั้งต้นของทุก notebook: mount Drive, clone repo (ใช้ `GH_PAT` จาก Colab Secrets), ทำ symlink ของ `data/` และ `app/models/forecast_v3/` ไปที่ Drive, และติดตั้ง `requirements-train.txt`. รันซ้ำได้โดยปลอดภัย.

Required Colab userdata secrets / secrets ที่ต้องตั้ง:

- `GH_PAT` — GitHub fine-grained personal access token (scope: read repo)
- `CDSAPI_KEY` — สำหรับ ERA5 ingest (optional ใน notebook นี้)
- `TMD_API_KEY` — สำหรับ TMD ingest (optional ใน notebook นี้)


In [ ]:
# --- bootstrap (clone + drive + pip) ---------------------------------------
import os, urllib.request
from google.colab import userdata

# Read secrets (fail-soft so the cell still runs in dev mode).
for key in ("GH_PAT", "CDSAPI_KEY", "TMD_API_KEY"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception as exc:
        print(f"[setup] secret {key} not set: {exc}")

# GitHub coordinates: orbitorls/HeatShield
os.environ.setdefault("GH_OWNER", "orbitorls")
os.environ.setdefault("GH_REPO", "HeatShield")
os.environ.setdefault("GH_BRANCH", "main")

REPO_DIR = "/content/Heat-wave-backend"
BOOTSTRAP = f"{REPO_DIR}/scripts/colab_bootstrap.sh"

# Pull bootstrap script before the repo is cloned.
if not os.path.exists(BOOTSTRAP):
    raw_url = (
        f"https://raw.githubusercontent.com/{os.environ['GH_OWNER']}/"
        f"{os.environ['GH_REPO']}/{os.environ['GH_BRANCH']}/scripts/colab_bootstrap.sh"
    )
    tmp = "/tmp/colab_bootstrap.sh"
    try:
        urllib.request.urlretrieve(raw_url, tmp)
        BOOTSTRAP = tmp
    except Exception as exc:
        print(f"[setup] could not pre-fetch bootstrap ({exc}); will rely on the repo copy.")

!bash {BOOTSTRAP}


In [ ]:
# --- import test + GPU check ----------------------------------------------
import sys, importlib

for mod in ("numpy", "pandas", "xgboost", "lightgbm", "sklearn", "optuna"):
    try:
        m = importlib.import_module(mod)
        print(f"{mod:<10s} {getattr(m, '__version__', '?')}")
    except ImportError as exc:
        print(f"{mod:<10s} MISSING ({exc})")

# torch is optional — only relevant for TabPFN backend.
try:
    import torch
    print(f"torch      {torch.__version__} | cuda available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"           device: {torch.cuda.get_device_name(0)}")
except Exception as exc:
    print(f"torch      not installed ({exc.__class__.__name__}); GPU check skipped.")

print(f"python     {sys.version.split()[0]}")


In [ ]:
# --- verify Drive mount + repo + station registry -------------------------
import os, sys

REPO_DIR = "/content/Heat-wave-backend"
DRIVE_ROOT = "/content/drive/MyDrive/heatshield"

checks = {
    "Drive mounted":      os.path.isdir("/content/drive/MyDrive"),
    "Drive heatshield/":  os.path.isdir(DRIVE_ROOT),
    "Repo cloned":        os.path.isdir(REPO_DIR),
    "data/ symlink":      os.path.islink(f"{REPO_DIR}/data"),
    "models/forecast_v3 symlink": os.path.islink(f"{REPO_DIR}/app/models/forecast_v3"),
}
for label, ok in checks.items():
    print(f"  [{'OK' if ok else '--'}] {label}")

# Make repo importable for subsequent cells.
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from app.data.stations import STATIONS
print(f"\nStations registered: {list(STATIONS.keys())}")
assert set(STATIONS.keys()) == {"BKK_01", "CNX_01", "KKN_01", "HYI_01", "RYG_01"}, \
    "Station registry mismatch — see app/data/stations.py"
print("Station registry OK.")


## Summary / สรุป

**EN.** If every check above shows `[OK]`, the Colab environment is ready. Drive holds the persistent `data/` and `app/models/forecast_v3/` directories so subsequent notebooks can ingest, train, and push artifacts without re-downloading. Open `01_ingest.ipynb` next.

**TH.** ถ้าตัวตรวจทุกข้อขึ้น `[OK]` แสดงว่าสภาพแวดล้อม Colab พร้อมใช้งาน. Drive จะเก็บ `data/` และ `app/models/forecast_v3/` แบบถาวรเพื่อให้ notebook ถัดไป ingest / train / push artifacts ได้โดยไม่ต้องโหลดข้อมูลใหม่. ขั้นถัดไปคือเปิด `01_ingest.ipynb`.

**Troubleshooting / แก้ปัญหาเบื้องต้น**

- `GH_PAT` missing → ใส่ใน Colab → key icon (Secrets).
- Drive ขอ permission ใหม่ทุกครั้ง → กด `Authorize` ซ้ำ; bootstrap คือ idempotent.
- Symlink ไม่ขึ้น → ลบ `data/`, `app/models/forecast_v3/` ในกรณี local เก่าค้าง แล้วรัน Cell 1 ซ้ำ.
